In [456]:
import nflreadpy as nfl
import pandas as pd
import numpy as np

In [457]:
SEASON = 2025
pbp = nfl.load_pbp([SEASON]).to_pandas()
schedule = nfl.load_schedules([SEASON]).to_pandas()

In [458]:
scrimmage_plays = pbp.loc[
    pbp["posteam"].notna()
    & pbp["defteam"].notna()
    & pbp["epa"].notna()
    & pbp["success"].notna()
    & (pbp["pass"].eq(1) | pbp["rush"].eq(1))
    & pbp["qb_kneel"].fillna(0).ne(1)
    & pbp["qb_spike"].fillna(0).ne(1)
    & pbp["two_point_attempt"].fillna(0).ne(1)
].copy()

In [459]:
offense = (
    scrimmage_plays
    .groupby(['game_id', 'posteam'])
    .agg(
        off_epa = ('epa', 'mean'),
        off_success_rate = ('success', 'mean')
    )
    .reset_index()
)

In [460]:
pass_df = (
    scrimmage_plays.loc[scrimmage_plays["pass"].eq(1)]
    .groupby(['game_id', 'posteam'])
    .agg(pass_epa = ('epa', 'mean'))
    .reset_index()
)
rush_df = (
    scrimmage_plays.loc[scrimmage_plays["rush"].eq(1)]
    .groupby(['game_id', 'posteam'])
    .agg(rush_epa = ('epa', 'mean'))
    .reset_index()
)

In [461]:
offense_df = (
    offense
    .merge(pass_df, on=['game_id', 'posteam'], how='left')
    .merge(rush_df, on=['game_id', 'posteam'], how='left')
    .rename(columns={'posteam': 'team'})
)
offense_df.head()

,game_id,team,off_epa,off_success_rate,pass_epa,rush_epa
0,2025_01_ARI_NO,ARI,0.061773,0.424242,0.131530,-0.052630
1,2025_01_ARI_NO,NO,-0.057140,0.424658,-0.014034,-0.157067
2,2025_01_BAL_BUF,BAL,0.394435,0.509804,0.530099,0.282997
3,2025_01_BAL_BUF,BUF,0.302612,0.512821,0.457560,-0.046021
4,2025_01_CAR_JAX,CAR,-0.176198,0.365079,-0.214952,-0.092878


In [462]:
defense_df = (
    scrimmage_plays
    .groupby(['game_id', 'defteam'])
    .agg(
        def_epa_allowed = ('epa', 'mean'),
        def_success_rate_allowed = ('success', 'mean'),
    )
    .reset_index()
    .rename(columns={'defteam': 'team'})
)
defense_df.head()

,game_id,team,def_epa_allowed,def_success_rate_allowed
0,2025_01_ARI_NO,ARI,-0.057140,0.424658
1,2025_01_ARI_NO,NO,0.061773,0.424242
2,2025_01_BAL_BUF,BAL,0.302612,0.512821
3,2025_01_BAL_BUF,BUF,0.394435,0.509804
4,2025_01_CAR_JAX,CAR,0.103053,0.460317


In [463]:
schedule_df = schedule.loc[
    schedule['home_score'].notna()
    & schedule['away_score'].notna(),
    ['game_id', 'season', 'week', 'gameday', 'game_type', 'home_team', 'away_team', 'home_score', 'away_score', 'home_rest', 'away_rest', 'location'],
].copy()

In [464]:
schedule_df['home_win'] = np.where(
    schedule_df['home_score'].eq(schedule_df['away_score']), np.nan,
    schedule_df['home_score'].gt(schedule_df['away_score']).astype(int)
)

In [465]:
schedule_df['neutral_site'] = (schedule_df['location'].astype(str).str.lower().eq('neutral')).astype(int)

In [466]:
home_rows = schedule_df[['season', 'week', 'gameday', 'game_id', 'game_type', 'neutral_site']].copy()

home_rows['team'] = schedule_df['home_team']
home_rows['opponent'] = schedule_df['away_team']
home_rows['points_scored'] = schedule_df['home_score']
home_rows['points_allowed'] = schedule_df['away_score']
home_rows['rest_days'] = schedule_df['home_rest']
home_rows['is_home'] = 1

away_rows = schedule_df[['season', 'week', 'gameday', 'game_id', 'game_type', 'neutral_site']].copy()

away_rows['team'] = schedule_df['away_team']
away_rows['opponent'] = schedule_df['home_team']
away_rows['points_scored'] = schedule_df['away_score']
away_rows['points_allowed'] = schedule_df['home_score']
away_rows['rest_days'] = schedule_df['away_rest']
away_rows['is_home'] = 0

In [467]:
def add_team_result(df):
    df = df.copy()

    df['win'] = np.select(
        [
            df['points_scored'].gt(df['points_allowed']),
            df['points_scored'].lt(df['points_allowed'])
        ],
        [1.0, 0.0],
        default=0.5,
    )
    df['home_field'] = (
    df['is_home'].eq(1) 
    & df['neutral_site'].eq(0)
    ).astype(int)

    return df

home_rows = add_team_result(home_rows)
away_rows = add_team_result(away_rows)

In [468]:
schedule_team_games_df = pd.concat([home_rows, away_rows], ignore_index=True)

In [469]:
team_games_df = (
    schedule_team_games_df
    .merge(
        offense_df, 
        on=['game_id', 'team'], 
        how='left',
    )
    .merge(
        defense_df, 
        on=['game_id', 'team'], 
        how='left',
    )    
)

In [470]:
team_games_df = (
    team_games_df
    .sort_values(['team', 'gameday', 'game_id'])
    .reset_index(drop=True)
)

In [471]:
team_games_df = team_games_df[[
    'season',
    'week',
    'gameday',  
    'game_id',
    'game_type',
    'team',
    'opponent',
    'is_home',
    'neutral_site',
    'home_field',
    'rest_days',
    'off_epa',
    'off_success_rate',
    'pass_epa',
    'rush_epa',
    'def_epa_allowed',
    'def_success_rate_allowed',
    'points_scored',
    'points_allowed',
    'win',
]]

In [472]:
team_games_df.loc[
    team_games_df["game_id"] == "2025_01_BAL_BUF"
]

,season,week,gameday,game_id,game_type,team,opponent,is_home,neutral_site,home_field,rest_days,off_epa,off_success_rate,pass_epa,rush_epa,def_epa_allowed,def_success_rate_allowed,points_scored,points_allowed,win
34,2025,1,2025-09-07,2025_01_BAL_BUF,REG,BAL,BUF,0,0,0,7,0.394435,0.509804,0.530099,0.282997,0.302612,0.512821,40,41,0.0
51,2025,1,2025-09-07,2025_01_BAL_BUF,REG,BUF,BAL,1,0,1,7,0.302612,0.512821,0.457560,-0.046021,0.394435,0.509804,41,40,1.0


In [473]:
team_games_df['pregame_off_epa'] = (
    team_games_df
    .groupby(['season', 'team'])['off_epa']
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [474]:
team_games_df = (
    team_games_df
    .sort_values(['team', 'gameday', 'game_id'])
    .reset_index(drop=True)
)

In [475]:
pregame_feature_map = {
    'off_epa': 'pregame_off_epa',
    'off_success_rate': 'pregame_off_success_rate',
    'pass_epa': 'pregame_pass_epa',
    'rush_epa': 'pregame_rush_epa',
    'def_epa_allowed': 'pregame_def_epa_allowed',
    'def_success_rate_allowed': 'pregame_def_success_rate_allowed',
    'points_scored': 'pregame_ppg',
    'points_allowed': 'pregame_ppg_allowed',
    'win': 'pregame_win_pct',
}


In [476]:
for raw_column, pregame_column in pregame_feature_map.items():
    team_games_df[pregame_column] = (
        team_games_df
        .groupby(['season', 'team'], sort=False)[raw_column]
        .transform(lambda x: x.shift(1).expanding(min_periods=1).mean())
    )

In [477]:
team_games_df['prior_games'] = (
    team_games_df
    .groupby(['season', 'team'], sort=False)
    .cumcount()
)

In [478]:
team_games_df.loc[
    (team_games_df['season'].eq(SEASON))
    & (team_games_df['team'].eq('BUF')),
    ['week', 'gameday', 'opponent', 'off_epa', 'pregame_off_epa', 'points_scored', 'pregame_ppg', 'win', 'pregame_win_pct','prior_games']
]

,week,gameday,opponent,off_epa,pregame_off_epa,points_scored,pregame_ppg,win,pregame_win_pct,prior_games
51,1,2025-09-07,BAL,0.302612,NaN,41,NaN,1.0,NaN,0
52,2,2025-09-14,NYJ,0.224746,0.302612,30,41.000000,1.0,1.000000,1
53,3,2025-09-18,MIA,0.260977,0.263679,31,35.500000,1.0,1.000000,2
54,4,2025-09-28,NO,0.162999,0.262778,31,34.000000,1.0,1.000000,3
55,5,2025-10-05,NE,0.002311,0.237834,20,33.250000,0.0,1.000000,4
56,6,2025-10-13,ATL,-0.104161,0.190729,14,30.600000,0.0,0.800000,5
57,8,2025-10-26,CAR,0.327073,0.141581,40,27.833333,1.0,0.666667,6
58,9,2025-11-02,KC,0.219664,0.168080,28,29.571429,1.0,0.714286,7
59,10,2025-11-09,MIA,-0.069657,0.174528,13,29.375000,0.0,0.750000,8
60,11,2025-11-16,TB,0.302729,0.147396,44,27.555556,1.0,0.666667,9


In [479]:
pregame_columns = list(pregame_feature_map.values())

first_team_games = (
    team_games_df
    .groupby(["season", "team"], sort=False)
    .head(1)
)

assert first_team_games[pregame_columns].isna().all().all()

In [480]:
team_feature_columns = [
    'game_id', 'team', 'prior_games', *pregame_columns,
]
pregame_team_features_df = team_games_df[team_feature_columns].copy()

In [481]:
home_features_df = pregame_team_features_df.rename(
    columns={
        'team': 'home_team',
        'prior_games': 'home_prior_games',
        **{
            column: f"home_{column}"
            for column in pregame_columns
        },
    }
)

In [482]:
away_features_df = pregame_team_features_df.rename(
    columns={
        'team': 'away_team',
        'prior_games': 'away_prior_games',
        **{
            column: f"away_{column}"
            for column in pregame_columns
        },
    }
)

In [483]:
game_columns = [
    'game_id', 'season', 'week', 'gameday', 'game_type', 'home_team', 'away_team', 'neutral_site', 'home_rest', 'away_rest', 'home_win'
]
games = schedule_df[game_columns].copy()

In [484]:
games = games.rename(
    columns={
        'home_rest': 'home_rest_days',
        'away_rest': 'away_rest_days',
    }
)
games['home_field'] = 1 - games['neutral_site']

In [ ]:
model_df = (
    games
    .merge(home_features_df, on=['game_id', 'home_team'], how='left')
    .merge(away_features_df, on=['game_id', 'away_team'], how='left')
)
model_df.tail()

,game_id,season,week,gameday,game_type,home_team,away_team,neutral_site,home_rest_days,away_rest_days,...,away_prior_games,away_pregame_off_epa,away_pregame_off_success_rate,away_pregame_pass_epa,away_pregame_rush_epa,away_pregame_def_epa_allowed,away_pregame_def_success_rate_allowed,away_pregame_ppg,away_pregame_ppg_allowed,away_pregame_win_pct
280,2025_20_LA_CHI,2025,20,2026-01-18,DIV,CHI,LA,0,8,8,...,18,0.136963,0.518580,0.210363,0.002152,-0.063124,0.417304,30.666667,20.944444,0.722222
281,2025_20_HOU_NE,2025,20,2026-01-18,DIV,NE,HOU,0,7,6,...,18,-0.007856,0.420437,0.086249,-0.154848,-0.138963,0.387058,24.111111,16.722222,0.722222
282,2025_21_NE_DEN,2025,21,2026-01-25,CON,DEN,NE,0,8,7,...,19,0.134316,0.472426,0.300832,-0.102414,-0.070397,0.441251,28.105263,17.842105,0.842105
283,2025_21_LA_SEA,2025,21,2026-01-25,CON,SEA,LA,0,8,7,...,19,0.132163,0.511108,0.201652,0.004522,-0.061685,0.420032,30.105263,20.736842,0.736842
284,2025_22_SEA_NE,2025,22,2026-02-08,SB,NE,SEA,1,14,14,...,19,0.067252,0.473419,0.205478,-0.073110,-0.130654,0.398822,29.210526,17.105263,0.842105
